# KaraokeSep na nuvem

Separa as faixas na placa de vídeo do Google, de graça, e devolve um zip pronto
para descompactar na pasta `saida` do seu KaraokeSep.

---

## Só isto: dois cliques

**1.** Menu `Ambiente de execução` → `Alterar o tipo de ambiente de execução` →
acelerador **T4 GPU** → Salvar.

**2.** Escolha o modo abaixo e clique no ▶ da célula.

Quando aparecer o botão **Escolher arquivos**, selecione as músicas. O resto
acontece sozinho: instala, separa, empacota e baixa.

---

| | No seu notebook | Aqui |
|---|---|---|
| 4 faixas | htdemucs | BS-Roformer-SW |
| Tempo por música | ~5 min | menos de 1 min |
| Modo Profundo | mais de 2 h | ~3 min |

A placa não muda a qualidade, muda a velocidade — e com ela os modelos pesados
ficam viáveis, o que na prática melhora o resultado.

In [ ]:
#@markdown ### Configuração
MODO = "padrao"  #@param ["karaoke", "padrao", "padrao_6", "profundo"]
FORMATO = "FLAC"  #@param ["FLAC", "MP3", "WAV"]
REFAZER = False  #@param {type:"boolean"}
SEPARAR_VOZES = False  #@param {type:"boolean"}
PARTITURA = "voz_e_baixo"  #@param ["nenhuma", "voz_e_baixo"]
#@markdown **karaoke** 2 faixas · **padrao** 4 faixas · **padrao_6** 6 faixas ·
#@markdown **profundo** 10 faixas
#@markdown
#@markdown FLAC guarda sem perda. MP3 320 baixa em um terço do tamanho.
#@markdown
#@markdown `REFAZER` separa de novo o que já foi separado nesta sessão.
#@markdown
#@markdown `SEPARAR_VOZES` divide a voz em masculina e feminina, para dueto.
#@markdown Funciona quando cada um canta um trecho; falha quando cantam juntos
#@markdown ou quando os timbres são parecidos. Custa uma passada a mais.
#@markdown
#@markdown `PARTITURA` gera partitura (MusicXML, PDF e MIDI) da voz e do baixo.
#@markdown Se ligar, rode também a célula **Partitura** logo abaixo: é ela que
#@markdown empacota e baixa o zip.

import os, subprocess, sys, time
from pathlib import Path

PASSO = 0
def etapa(texto):
    global PASSO
    PASSO += 1
    print(f"\n{'='*62}\n  {PASSO}. {texto}\n{'='*62}")

# ------------------------------------------------------------------ #
etapa("Conferindo a placa de video")
# ------------------------------------------------------------------ #
r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                    "--format=csv,noheader"], capture_output=True, text=True)
if r.returncode != 0 or not r.stdout.strip():
    print("   NENHUMA PLACA ATIVA.\n")
    print("   Menu: Ambiente de execucao > Alterar o tipo de ambiente")
    print("   Escolha T4 GPU, salve, e rode esta celula de novo.")
    raise SystemExit("sem placa")
print("   ", r.stdout.strip())

# ------------------------------------------------------------------ #
etapa("Instalando")
# ------------------------------------------------------------------ #
try:
    import audio_separator, audioread           # noqa: F401
    print("   Ja instalado nesta sessao.")
except ImportError:
    print("   Sao mais de 2 GB. De 2 a 4 minutos. Nao feche a aba.")
    t0 = time.time()
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "audio-separator[gpu]", "audioread"], check=True)
    print(f"   Pronto em {time.time()-t0:.0f} s.")

import torch
print(f"   PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise SystemExit("CUDA indisponivel; confira o acelerador")

# ------------------------------------------------------------------ #
etapa("Enviando as musicas")
# ------------------------------------------------------------------ #
EXT = {".mp3",".flac",".wav",".m4a",".ogg",".opus",".aac",".wma",".aiff"}
ENTRADA = Path("/content/entrada"); ENTRADA.mkdir(exist_ok=True)

def listar():
    return sorted(f for f in ENTRADA.iterdir()
                  if f.is_file() and f.suffix.lower() in EXT)

if listar():
    print(f"   {len(listar())} musica(s) ja enviadas nesta sessao; reaproveitando.")
    print("   Para trocar, apague a pasta /content/entrada e rode de novo.")
else:
    print("   Clique em ESCOLHER ARQUIVOS e selecione as musicas.")
    print("   Segure Ctrl para marcar varias. Nao feche a aba durante o envio.")
    import shutil as _sh
    from google.colab import files
    for nome in files.upload():
        _sh.move(nome, ENTRADA/nome)

musicas = listar()
if not musicas:
    raise SystemExit("nenhuma musica enviada")
for f in musicas:
    print(f"   {f.name}  ({f.stat().st_size/1e6:.1f} MB)")

# ------------------------------------------------------------------ #
etapa(f"Separando em modo {MODO}")
# ------------------------------------------------------------------ #
import functools, re, shutil
from audio_separator.separator import Separator

SAIDA = Path("/content/saida"); SAIDA.mkdir(exist_ok=True)
MODELOS = Path("/content/modelos"); MODELOS.mkdir(exist_ok=True)

CANDIDATOS = {
    "voz":     ["model_bs_roformer_ep_317_sdr_12.9755.ckpt",
                "melband_roformer_inst_v1e_plus.ckpt",
                "MDX23C-8KFFT-InstVoc_HQ.ckpt",
                "UVR-MDX-NET-Inst_HQ_4.onnx"],
    "stems":   ["BS-Roformer-SW.ckpt", "htdemucs_6s.yaml", "htdemucs_ft.yaml"],
    "drumsep": ["MDX23C-DrumSep-aufr33-jarredou.ckpt"],
    "backing": ["mel_band_roformer_karaoke_aufr33_viperx_sdr_10.1956.ckpt",
                "bs_roformer_karaoke_anvuew.ckpt"],
    # voz masculina x feminina. SDR 7.29, bem abaixo dos stems instrumentais:
    # espere vazamento audivel onde os dois cantam ao mesmo tempo.
    "vozes":   ["bs_roformer_male_female_by_aufr33_sdr_7.2889.ckpt",
                "male_female"],
}
MAPA = {"vocals":"vocal","vocal":"vocal","lead vocals":"vocal_principal",
        "instrumental":"instrumental","no vocals":"instrumental",
        "drums":"bateria","bass":"baixo","guitar":"guitarra","piano":"piano",
        "other":"outros","back vocals":"coro","backing vocals":"coro",
        "kick":"bumbo","snare":"caixa","toms":"toms","cymbals":"pratos",
        "hh":"chimbal","hihat":"chimbal","ride":"conducao","crash":"crash",
        "male":"voz_masculina","female":"voz_feminina"}

sep = Separator(log_level=40, model_file_dir=str(MODELOS), output_format=FORMATO)

def normalizar(s): return re.sub(r"[-_.\s]+", "", s.lower())

@functools.lru_cache(maxsize=1)
def catalogo():
    achados = set()
    def andar(no):
        if isinstance(no, str):
            b = no.lower()
            if (b.endswith((".ckpt",".onnx",".pth",".yaml")) and "://" not in no
                    and not b.startswith("config_") and "_config." not in b):
                achados.add(no)
        elif isinstance(no, dict):
            for k, v in no.items(): andar(k); andar(v)
        elif isinstance(no, (list, tuple, set)):
            for v in no: andar(v)
    andar(sep.list_supported_model_files())
    return sorted(achados)

def resolver(papel):
    disp = {normalizar(m): m for m in catalogo()}
    for cand in CANDIDATOS[papel]:
        alvo = normalizar(cand)
        if alvo in disp: return disp[alvo]
        for chave, orig in disp.items():
            if alvo in chave: return orig
    return None

def classificar(caminho):
    m = re.findall(r"\(([^)]+)\)", Path(caminho).stem)
    bruto = re.sub(r"[_-]+", " ", (m[-1] if m else "").strip().lower()).strip()
    if bruto in MAPA: return MAPA[bruto]
    for k, v in MAPA.items():
        if k in bruto: return v
    return re.sub(r"[^a-z0-9]+", "_", bruto) or "faixa"

def passada(entrada, papel, destino, criados, mapa=None):
    modelo = resolver(papel)
    if not modelo: raise RuntimeError(f"sem modelo para o papel {papel}")
    print(f"      modelo: {modelo}")
    bruto = destino/"_bruto"; bruto.mkdir(parents=True, exist_ok=True)
    antes = set(bruto.glob("*"))
    sep.output_dir = str(bruto)
    sep.load_model(model_filename=modelo)
    sep.separate(str(entrada))
    resultado = {}
    for origem in [p for p in bruto.glob("*") if p not in antes and p.is_file()]:
        nome = classificar(origem)
        if mapa: nome = mapa.get(nome, nome)
        alvo = destino/f"{nome}{origem.suffix}"
        # Uma passada da cascata nao pode apagar o que a anterior criou; mas
        # isso so vale para ESTA execucao, senao reprocessar duplica tudo.
        if alvo.resolve() == Path(entrada).resolve() or alvo.resolve() in criados:
            raiz = Path(entrada).stem.lower()
            nome = f"{raiz}_{nome}" if not nome.startswith(raiz) else f"{nome}_2"
            alvo = destino/f"{nome}{origem.suffix}"
            print(f"      conflito de nome, gravando como {alvo.name}")
        if alvo.exists(): alvo.unlink()
        shutil.move(str(origem), str(alvo))
        criados.add(alvo.resolve())
        resultado[nome] = alvo
    shutil.rmtree(bruto, ignore_errors=True)
    return resultado

def limpar_nome(t):
    t = re.sub(r"(?i)\s*(?:tuberipper|y2mate|ytmp3|savefrom|mp3juices|onlymp3)[a-z0-9._-]*", " ", t)
    t = re.sub(r"(?i)\s*\.(?:net|com|com\.br|org|io|cc|to|me)\b", " ", t)
    t = re.sub(r"(?i)\s*[\(\[][^\)\]]*(?:official|video|audio|lyric|hd|4k)[^\)\]]*[\)\]]", " ", t)
    t = re.sub(r"^\s*\d{1,3}\s*[-._)]\s*", "", t)
    t = re.sub(r'[<>:"/\\|?*]', "_", t)
    return re.sub(r"\s{2,}", " ", t).strip(" -._") or "musica"

feitas, puladas, falharam = 0, 0, []
for i, origem in enumerate(musicas, 1):
    nome = limpar_nome(origem.stem)
    pasta = SAIDA/nome
    ja = pasta.is_dir() and any(f.suffix.lower() in EXT for f in pasta.iterdir())
    if ja and not REFAZER:
        print(f"\n[{i}/{len(musicas)}] {nome}  —  ja separada, pulando")
        puladas += 1
        continue

    shutil.rmtree(pasta, ignore_errors=True)
    pasta.mkdir(parents=True, exist_ok=True)
    print(f"\n[{i}/{len(musicas)}] {nome}")
    t0 = time.time(); criados = set()
    try:
        if MODO == "karaoke":
            res = passada(origem, "voz", pasta, criados)
        else:
            res = passada(origem, "stems", pasta, criados)
            if MODO == "profundo":
                if "bateria" in res:
                    print("   dividindo a bateria...")
                    res.update(passada(res["bateria"], "drumsep", pasta, criados))
                if "vocal" in res:
                    print("   separando voz principal e coro...")
                    res.update(passada(res["vocal"], "backing", pasta, criados,
                               mapa={"vocal":"vocal_principal","instrumental":"coro"}))
        if SEPARAR_VOZES:
            alvo = res.get("vocal_principal") or res.get("vocal")
            if alvo:
                print("   dividindo a voz em masculina e feminina...")
                res.update(passada(alvo, "vozes", pasta, criados))
            else:
                print("   sem faixa de voz; separacao de vozes pulada")
        print(f"   {time.time()-t0:.0f} s  ->  {len(res)} faixas: "
              f"{', '.join(sorted(res))}")
        feitas += 1
    except Exception as e:
        import traceback; traceback.print_exc()
        falharam.append(nome)

# ------------------------------------------------------------------ #
etapa("Conferindo o resultado")
# ------------------------------------------------------------------ #
total = 0.0
for pasta in sorted(SAIDA.iterdir()):
    if not pasta.is_dir(): continue
    arq = sorted(f for f in pasta.iterdir() if f.is_file())
    mb = sum(f.stat().st_size for f in arq)/1e6
    total += mb
    print(f"   [{pasta.name}]  {len(arq)} faixas, {mb:.0f} MB")
    print(f"      {', '.join(f.stem for f in arq)}")
print(f"\n   {feitas} separadas, {puladas} puladas"
      + (f", {len(falharam)} com erro: {', '.join(falharam)}" if falharam else ""))
print(f"   Total em disco: {total:.0f} MB")

# ------------------------------------------------------------------ #
etapa("Baixando")
# ------------------------------------------------------------------ #
if PARTITURA != "nenhuma" and total > 0:
    print("   PARTITURA ligada: o zip sai na celula Partitura, logo abaixo.")
    print("   Rode a proxima celula para gerar as partituras e baixar tudo.")
elif total == 0:
    print("   Nada para baixar.")
else:
    if total > 400:
        print("   AVISO: acima de 400 MB o download costuma falhar.")
        print("   Se cair, rode de novo com FORMATO = MP3.")
    shutil.make_archive("/content/stems", "zip", str(SAIDA))
    tam = Path("/content/stems.zip").stat().st_size/1e6
    print(f"   stems.zip ({tam:.0f} MB) — o download comeca sozinho.")
    from google.colab import files as _f
    _f.download("/content/stems.zip")

if PARTITURA == "nenhuma":
    print(f"\n{'='*62}")
    print("  Pronto. Descompacte o stems.zip dentro da pasta 'saida'")
    print("  do KaraokeSep e abra o festa.bat.")
    print(f"{'='*62}")


  1. Conferindo a placa de video
    Tesla T4, 15360 MiB

  2. Instalando
   Sao mais de 2 GB. De 2 a 4 minutos. Nao feche a aba.
   Pronto em 32 s.
   PyTorch 2.11.0+cu130 | CUDA: True

  3. Enviando as musicas
   Clique em ESCOLHER ARQUIVOS e selecione as musicas.
   Segure Ctrl para marcar varias. Nao feche a aba durante o envio.


---

## Partitura da voz e do baixo

Rode esta célula **depois** da separação, na mesma sessão, com
`PARTITURA = "voz_e_baixo"` ligada na célula anterior. Para cada música ela
gera, dentro da pasta da música, `partituras/` com:

| Arquivo | O que é |
|---|---|
| `<faixa>.musicxml` | a partitura, abre no MuseScore e em qualquer editor |
| `<faixa>.pdf` | pronta para imprimir |
| `<faixa>.mid` | as notas detectadas, com o tempo real do áudio |
| `<faixa>.svg` e `<faixa>.timemap.json` | a partitura na tela do modo festa, com o cursor |
| `partitura.json` | batidas, compasso, tonalidade e confiança por compasso |

O ritmo segue as batidas detectadas na música, não um BPM fixo. A confiança por
compasso compara o que foi escrito com o que se ouve na faixa: serve de alarme
para os trechos que precisam de correção, não é garantia de acerto.

Se o Beat This! não baixar, ela usa o rastreador do librosa e avisa na tela.
A célula termina baixando o `stems.zip` já com as partituras dentro.

In [ ]:
#@markdown ### Partitura: voz e baixo
#@markdown Roda depois da separação, na mesma sessão. Gera, para cada música, a
#@markdown pasta `partituras` com MusicXML, PDF e MIDI de cada faixa, mais o
#@markdown `partitura.json` que o modo festa lê. Escolha `PARTITURA` na célula
#@markdown anterior.

import hashlib, json, math, re, shutil, subprocess, sys, time
from datetime import datetime
from pathlib import Path

import numpy as np

VERSAO_FORMATO = 2
ENTRADA = Path("/content/entrada")
SAIDA = Path("/content/saida")
EXT = {".mp3", ".flac", ".wav", ".m4a", ".ogg", ".opus", ".aac", ".wma", ".aiff"}

# Pasta de cada faixa -> como entra na partitura. A voz principal tem
# prioridade sobre a voz inteira: transcrever as duas daria o mesmo canto duas
# vezes. Vozes de dueto sao transcritas junto, se existirem.
PAPEIS = {
    "vocal_principal": ("voz", "Voz"),
    "vocal":           ("voz", "Voz"),
    "voz_masculina":   ("voz", "Voz masculina"),
    "voz_feminina":    ("voz", "Voz feminina"),
    "baixo":           ("baixo", "Baixo"),
}

# fmin do baixo: o CREPE so cobre a partir de 32,7 Hz; abaixo disso devolve -inf.
# Limiares ainda nao calibrados em musica real: ficam aqui, num lugar so,
# para ajustar depois de ouvir o primeiro resultado.
AJUSTES = {
    "voz":   dict(fmin=65.0, fmax=1100.0, periodicidade=0.35, queda_db=35.0,
                  salto_st=0.8, salto_quadros=5, nota_min=0.06, junta=0.06),
    "baixo": dict(fmin=32.7, fmax=400.0, periodicidade=0.30, queda_db=35.0,
                  salto_st=0.8, salto_quadros=5, nota_min=0.05, junta=0.05),
}
PASSO_CREPE = 0.01          # segundos por quadro (hop de 160 amostras a 16 kHz)


def etapa(texto):
    global _PASSO
    _PASSO = globals().get("_PASSO", 0) + 1
    print(f"\n{'='*62}\n  {_PASSO}. {texto}\n{'='*62}")


def instalar():
    try:
        import torchcrepe, beat_this, music21, verovio, cairosvg, pypdf  # noqa
        import librosa, pretty_midi                                        # noqa
        print("   Ja instalado nesta sessao.")
        return
    except ImportError:
        pass
    print("   De 1 a 3 minutos.")
    t0 = time.time()
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "torchcrepe", "music21",
         "verovio", "cairosvg", "pypdf", "librosa", "pretty_midi",
         "beat_this"],
        capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-1500:], r.stderr[-1500:])
        raise SystemExit("falha ao instalar as bibliotecas de partitura")
    print(f"   Pronto em {time.time()-t0:.0f} s.")


# --------------------------------------------------------------------------- #
#  Grade de batidas
#
#  A partitura NAO usa um BPM fixo. Cada batida detectada vira uma seminima e
#  o instante real de cada uma fica em partitura.json ("batidas"). No navegador
#  o "qstamp" do timemap do Verovio e o indice fracionario da batida (unidade:
#  seminima, independe de andamento); o player o converte no instante do audio
#  por interpolacao entre as batidas.
# --------------------------------------------------------------------------- #
def montar_grade(batidas, tempos_fortes, duracao):
    """Devolve (grade, por_compasso). grade[k] = instante da batida k da
    partitura; a batida 0 abre um compasso."""
    b = np.asarray(batidas, float)
    d = np.asarray(tempos_fortes, float)
    if len(b) < 4:
        raise ValueError("batidas insuficientes para montar a grade")
    inter = np.diff(b)
    iv0 = float(np.median(inter[:8]))
    iv1 = float(np.median(inter[-8:]))

    idx = np.array([int(np.argmin(np.abs(b - x))) for x in d]) if len(d) else []
    por_compasso = 4
    if len(idx) >= 3:
        cand = int(round(float(np.median(np.diff(idx)))))
        if 2 <= cand <= 7:
            por_compasso = cand
    primeiro = int(idx[0]) if len(idx) else 0

    # batidas que faltam antes do primeiro tempo forte para fechar um compasso
    # (anacruse): a grade comeca nelas, extrapoladas com o primeiro intervalo
    inicio_idx = primeiro - por_compasso * math.ceil(primeiro / por_compasso)

    lista = []
    if inicio_idx < 0:
        # extrapola para tras com o primeiro intervalo
        lista += [b[0] - iv0 * k for k in range(-inicio_idx, 0, -1)]
        corte = 0
    else:
        corte = inicio_idx
    lista += list(b[corte:])
    # extrapola para frente ate cobrir a musica inteira e fechar o compasso
    while lista[-1] < duracao + iv1 or len(lista) % por_compasso:
        lista.append(lista[-1] + iv1)
    return np.array(lista), por_compasso


def para_batida(t, grade):
    return np.interp(t, grade, np.arange(len(grade)))


def para_tempo(pos, grade):
    return np.interp(pos, np.arange(len(grade)), grade)


# --------------------------------------------------------------------------- #
#  Linha melodica: f0 -> notas
# --------------------------------------------------------------------------- #
def f0_quadros(y, sr, aj, dispositivo):
    import torch, torchcrepe, librosa
    audio = torch.tensor(y, dtype=torch.float32)[None]
    passo = int(round(PASSO_CREPE * sr))
    freq, per = torchcrepe.predict(
        audio, sr, passo, aj["fmin"], aj["fmax"], model="full",
        decoder=torchcrepe.decode.viterbi, return_periodicity=True,
        batch_size=1024, device=dispositivo)
    per = torchcrepe.filter.median(per, 3)
    freq = torchcrepe.filter.mean(freq, 3)
    freq, per = freq[0].cpu().numpy(), per[0].cpu().numpy()

    rms = librosa.feature.rms(y=y, frame_length=1024, hop_length=passo,
                              center=True)[0]
    n = min(len(freq), len(rms))
    freq, per, rms = freq[:n], per[:n], rms[:n]
    db = 20 * np.log10(np.maximum(rms, 1e-8))
    ref = np.percentile(db, 95)
    voz = (per >= aj["periodicidade"]) & (db >= ref - aj["queda_db"]) \
        & (db > -70) & np.isfinite(freq) & (freq > 0)
    midi = 69 + 12 * np.log2(np.maximum(freq, 1e-3) / 440.0)
    desvio = afinacao(midi, voz)
    return midi - desvio, voz, desvio


def afinacao(midi, voz):
    """Desvio medio de afinacao do audio, em semitons (-0,5 a 0,5). Disco
    gravado em 432 Hz ou voz sistematicamente alta joga cada nota para o
    meio entre duas teclas; arredondar sem corrigir erra a nota."""
    m = midi[voz]
    if len(m) < 50:
        return 0.0
    frac = m - np.round(m)
    return float(np.angle(np.mean(np.exp(2j * np.pi * frac))) / (2 * np.pi))


def corrigir_oitavas(notas, janela=4):
    """O CREPE as vezes salta uma oitava por algumas dezenas de milissegundos.
    Uma nota curta que destoa dos vizinhos por quase exatamente uma oitava
    volta para o registro deles. Salto longo e melodia de verdade: fica."""
    ms = [n[2] for n in notas]
    saida = []
    for i, (a, b, m) in enumerate(notas):
        viz = ms[max(0, i - janela):i] + ms[i + 1:i + 1 + janela]
        if len(viz) >= 3 and (b - a) < 0.5:
            d = m - float(np.median(viz))
            k = round(d / 12)
            if k != 0 and abs(d - 12 * k) <= 2:
                m -= 12 * k
        saida.append((a, b, m))
    return saida


def segmentar(midi, voz, aj, passo=PASSO_CREPE):
    """Quadros de pitch -> [(inicio_s, fim_s, nota_midi)]. Corta a nota quando
    o pitch muda de forma consistente; vibrato e ruido de um quadro nao cortam."""
    notas = []
    cur = []          # [(quadro, midi)]
    desvio = []       # quadros fora da faixa da nota atual
    vazio = 0
    max_vazio = int(round(0.08 / passo))

    def fechar(fim_quadro):
        nonlocal cur
        if cur:
            notas.append([cur[0][0], fim_quadro, float(np.median([m for _, m in cur]))])
        cur = []

    ultimo_voz = -1
    for i in range(len(midi)):
        if not voz[i]:
            if cur:
                vazio += 1
                if vazio > max_vazio:
                    fechar(ultimo_voz + 1)
                    desvio = []
            continue
        vazio = 0
        ultimo_voz = i
        m = float(midi[i])
        if not cur:
            cur = [(i, m)]
            desvio = []
            continue
        centro = float(np.median([x for _, x in cur[-15:]]))
        if abs(m - centro) < aj["salto_st"]:
            cur.append((i, m))
            desvio = []
            continue
        desvio.append((i, m))
        if len(desvio) >= aj["salto_quadros"]:
            vals = [x for _, x in desvio]
            if max(vals) - min(vals) < 0.8:      # novo pitch estavel
                fechar(desvio[0][0])
                cur = list(desvio)
                desvio = []
            else:                                # instavel: segue a nota
                desvio = desvio[1:]

    fechar(ultimo_voz + 1)

    saida = []
    for ini, fim, m in notas:
        if (fim - ini) * passo < aj["nota_min"]:
            continue
        saida.append([ini * passo, fim * passo, int(round(m))])
    # une notas iguais separadas por um buraco minusculo
    unidas = []
    for n in saida:
        if unidas and unidas[-1][2] == n[2] and n[0] - unidas[-1][1] <= aj["junta"]:
            unidas[-1][1] = n[1]
        else:
            unidas.append(n)
    return [tuple(n) for n in unidas]


# --------------------------------------------------------------------------- #
#  Quantizacao pela grade de batidas
# --------------------------------------------------------------------------- #
def quantizar(notas, grade, divisao=4):
    """[(ini_s, fim_s, midi)] -> [(inicio_em_seminimas, duracao, midi)].
    Sem tercinas: divisao em semicolcheias."""
    q = []
    for ini, fim, m in notas:
        a = round(float(para_batida(ini, grade)) * divisao) / divisao
        b = round(float(para_batida(fim, grade)) * divisao) / divisao
        if b <= a:
            b = a + 1 / divisao
        q.append([a, b, m])
    q.sort(key=lambda x: (x[0], -(x[1] - x[0])))

    saida = []
    for a, b, m in q:
        if saida and a < saida[-1][1]:
            if a == saida[-1][0]:
                continue                       # mesma entrada: fica a mais longa
            saida[-1][1] = a                   # monofonico: corta a anterior
        if saida and 0 < a - saida[-1][1] <= 1 / divisao:
            saida[-1][1] = a                   # buraco de uma semicolcheia: liga
        saida.append([a, b, m])
    return [(a, b - a, m) for a, b, m in saida if b - a > 0]


PERFIL_MAIOR = np.array([6.35, 2.23, 3.48, 2.33, 4.38, 4.09, 2.52, 5.19, 2.39, 3.66, 2.29, 2.88])
PERFIL_MENOR = np.array([6.33, 2.68, 3.52, 5.38, 2.60, 3.53, 2.54, 4.75, 3.98, 2.69, 3.34, 3.17])
TONICA_MAIOR = ["C", "D-", "D", "E-", "E", "F", "F#", "G", "A-", "A", "B-", "B"]
TONICA_MENOR = ["C", "C#", "D", "E-", "E", "F", "F#", "G", "G#", "A", "B-", "B"]


def tonalidade(q):
    from music21 import key
    h = np.zeros(12)
    for _, d, m in q:
        h[m % 12] += d
    if h.sum() == 0:
        return key.Key("C", "major")
    melhor, alvo = -2, ("C", "major")
    for t in range(12):
        for perfil, modo, nomes in ((PERFIL_MAIOR, "major", TONICA_MAIOR),
                                    (PERFIL_MENOR, "minor", TONICA_MENOR)):
            c = np.corrcoef(h, np.roll(perfil, t))[0, 1]
            if c > melhor:
                melhor, alvo = c, (nomes[t], modo)
    return key.Key(*alvo)


def acertar_acidentes(parte, sharps):
    """O music21 10.5 marca bequadros espurios a partir do terceiro compasso.
    Recalcula aqui, pela armadura e pelo que ja soou no compasso."""
    from music21 import key, pitch, stream
    da_armadura = {p.step: p.accidental.alter
                   for p in key.KeySignature(sharps).alteredPitches}
    for m in parte.getElementsByClass(stream.Measure):
        no_compasso = {}
        for n in m.notes:
            p = n.pitch
            alter = p.accidental.alter if p.accidental is not None else 0
            vigente = no_compasso.get((p.step, p.octave),
                                      da_armadura.get(p.step, 0))
            continuacao = n.tie is not None and n.tie.type in ("continue", "stop")
            mostrar = alter != vigente and not continuacao
            no_compasso[(p.step, p.octave)] = alter
            if alter == 0:
                p.accidental = pitch.Accidental(0) if mostrar else None
            else:
                p.accidental = pitch.Accidental(alter)
                p.accidental.displayStatus = mostrar
                if mostrar:
                    p.accidental.displayStatus = True


def montar_partitura(q, nome, papel, grade, por_compasso, titulo):
    from music21 import (stream, note, meter, key, clef, instrument, metadata)
    k = tonalidade(q)
    parte = stream.Part()
    parte.partName = nome
    if papel == "baixo":
        parte.insert(0, instrument.ElectricBass())
        parte.insert(0, clef.BassClef())
    else:
        parte.insert(0, instrument.Vocalist())
        mediana = float(np.median([m for _, _, m in q])) if q else 64
        parte.insert(0, clef.Treble8vbClef() if mediana < 57 else clef.TrebleClef())
    parte.insert(0, meter.TimeSignature(f"{por_compasso}/4"))
    parte.insert(0, key.KeySignature(k.sharps))
    for inicio, dur, m in q:
        n = note.Note(m)
        if k.sharps < 0 and n.pitch.accidental is not None \
                and n.pitch.accidental.alter > 0:
            n.pitch = n.pitch.getEnharmonic()
        n.quarterLength = dur
        parte.insert(inicio, n)
    # preenche com pausas REAIS (sem isso o exportador cria pausas invisiveis)
    parte.makeRests(fillGaps=True, inPlace=True)
    parte.makeNotation(inPlace=True)
    # O Verovio nao conta o tempo de pausa de compasso inteiro ("rest
    # measure=yes") no timemap: toda nota depois de um compasso vazio chegaria
    # adiantada no cursor. Pausas explicitas contam.
    for r in parte.recurse().getElementsByClass(note.Rest):
        r.fullMeasure = False
        r.style.hideObjectOnPrint = False

    acertar_acidentes(parte, k.sharps)

    partitura = stream.Score()
    partitura.metadata = metadata.Metadata(title=titulo)
    partitura.insert(0, parte)
    return partitura, k


# --------------------------------------------------------------------------- #
#  Confianca por compasso
#
#  Compara o perfil de classes de nota (12 posicoes) do que foi escrito com o
#  do audio do stem naquele compasso. Nao e medida de acerto: e um alarme
#  para os compassos onde a transcricao nao tem nada a ver com o que se ouve.
# --------------------------------------------------------------------------- #
def confianca_compassos(y, sr, q, grade, por_compasso):
    import librosa
    hop = 512
    chroma = librosa.feature.chroma_cqt(y=y, sr=sr, hop_length=hop)
    rms = librosa.feature.rms(y=y, hop_length=hop)[0]
    n = min(chroma.shape[1], len(rms))
    chroma, rms = chroma[:, :n], rms[:n]
    db = 20 * np.log10(np.maximum(rms, 1e-8))
    ref = np.percentile(db, 95)
    tq = np.arange(n) * hop / sr

    total = len(grade) // por_compasso
    saida = []
    for c in range(total):
        b0, b1 = c * por_compasso, (c + 1) * por_compasso
        t0, t1 = para_tempo(b0, grade), para_tempo(b1, grade)
        sel = (tq >= t0) & (tq < t1)
        ativo = bool(sel.any() and (db[sel] > ref - 35).mean() > 0.2)

        perfil = np.zeros(12)
        for ini, dur, m in q:
            sobra = min(ini + dur, b1) - max(ini, b0)
            if sobra > 0:
                perfil[m % 12] += sobra
        tem_nota = perfil.sum() > 0

        if not tem_nota and not ativo:
            saida.append(None)                 # compasso vazio dos dois lados
        elif tem_nota != ativo:
            saida.append(0.0)
        else:
            ref_c = chroma[:, sel].mean(axis=1)
            den = np.linalg.norm(perfil) * np.linalg.norm(ref_c)
            saida.append(float(np.dot(perfil, ref_c) / den) if den > 0 else 0.0)
    return saida


# --------------------------------------------------------------------------- #
#  PDF, MIDI
# --------------------------------------------------------------------------- #
def gerar_pdf(musicxml, pdf):
    import verovio, cairosvg
    from pypdf import PdfWriter, PdfReader
    import io
    tk = verovio.toolkit()
    tk.setOptions({"pageWidth": 2100, "pageHeight": 2970, "scale": 45,
                   "adjustPageHeight": False, "footer": "none",
                   "breaks": "auto", "pageMarginLeft": 80, "pageMarginRight": 80,
                   "pageMarginTop": 80, "pageMarginBottom": 80})
    if not tk.loadFile(str(musicxml)):
        raise RuntimeError("Verovio nao leu o MusicXML")
    escritor = PdfWriter()
    for p in range(1, tk.getPageCount() + 1):
        pdf_pag = cairosvg.svg2pdf(bytestring=tk.renderToSVG(p).encode("utf-8"))
        escritor.append(PdfReader(io.BytesIO(pdf_pag)))
    with open(pdf, "wb") as f:
        escritor.write(f)


def gerar_svg_timemap(musicxml, svg, timemap):
    """Partitura em UMA pagina SVG longa (para rolar na tela) e o mapa de
    quais notas soam em cada batida. SVG e timemap saem da MESMA instancia do
    Verovio: os ids dos elementos mudam a cada carga do arquivo."""
    import verovio
    tk = verovio.toolkit()
    tk.setOptions({"pageWidth": 1700, "pageHeight": 60000, "scale": 55,
                   "adjustPageHeight": True, "footer": "none", "header": "none",
                   "breaks": "auto", "pageMarginLeft": 40, "pageMarginRight": 40,
                   "pageMarginTop": 40, "pageMarginBottom": 40})
    if not tk.loadFile(str(musicxml)):
        raise RuntimeError("Verovio nao leu o MusicXML")
    texto = tk.renderToSVG(1)
    tk.renderToMIDI()                      # o timemap so existe depois disto
    ativos, eventos = set(), []
    for e in tk.renderToTimemap():
        for i in e.get("off", []):
            ativos.discard(i)
        for i in e.get("on", []):
            ativos.add(i)
        eventos.append([float(e["qstamp"]), sorted(ativos)])
    # eventos no mesmo qstamp viram um so (o ultimo estado vale)
    unicos = []
    for q, a in eventos:
        if unicos and unicos[-1][0] == q:
            unicos[-1][1] = a
        else:
            unicos.append([q, a])
    marca = hashlib.sha1(texto.encode("utf-8")).hexdigest()[:16]
    Path(svg).write_text(texto, encoding="utf-8")
    Path(timemap).write_text(json.dumps({"hash": marca, "eventos": unicos},
                                        separators=(",", ":")), encoding="utf-8")
    return marca


def gerar_midi(notas, papel, destino):
    import pretty_midi
    pm = pretty_midi.PrettyMIDI()
    inst = pretty_midi.Instrument(program=33 if papel == "baixo" else 53)
    for ini, fim, m in notas:
        inst.notes.append(pretty_midi.Note(velocity=90, pitch=int(m),
                                           start=float(ini), end=float(fim)))
    pm.instruments.append(inst)
    pm.write(str(destino))


# --------------------------------------------------------------------------- #
#  Uma musica
# --------------------------------------------------------------------------- #
def achar_original(pasta, limpar_nome):
    if not ENTRADA.exists():
        return None
    for f in ENTRADA.iterdir():
        if f.suffix.lower() in EXT and limpar_nome(f.stem) == pasta.name:
            return f
    return None


def somar_stems(pasta, destino):
    import soundfile as sf
    arqs = [f for f in sorted(pasta.iterdir())
            if f.suffix.lower() in EXT and f.parent.name != "partituras"]
    nomes = {f.stem.lower(): f for f in arqs}
    if "instrumental" in nomes and "vocal" in nomes:
        arqs = [nomes["instrumental"], nomes["vocal"]]
    mix, sr0 = None, None
    for f in arqs:
        d, sr = sf.read(str(f), always_2d=True)
        d = d.mean(axis=1)
        if mix is None:
            mix, sr0 = d, sr
        else:
            n = min(len(mix), len(d))
            mix = mix[:n] + d[:n]
    if mix is None:
        return None
    sf.write(str(destino), mix / max(1.0, np.abs(mix).max()), sr0)
    return destino


def grade_da_musica(pasta, limpar_nome, dispositivo):
    """Batidas e tempos fortes. Beat This! primeiro; se o modelo nao baixar
    ou falhar, cai no rastreador do librosa (sem tempos fortes: o compasso
    vira 4/4 a partir da primeira batida)."""
    import librosa
    origem = achar_original(pasta, limpar_nome)
    fonte = "original"
    tmp = pasta / "_mix_temp.wav"
    if origem is None:
        origem = somar_stems(pasta, tmp)
        fonte = "soma dos stems"
        if origem is None:
            raise RuntimeError("sem audio para achar as batidas")
    try:
        try:
            from beat_this.inference import File2Beats
            f2b = File2Beats(checkpoint_path="final0", device=dispositivo,
                             dbn=False)
            batidas, fortes = f2b(str(origem))
            fonte += " / Beat This!"
        except Exception as e:
            print(f"\n   AVISO: Beat This! indisponivel ({type(e).__name__}); "
                  "usando o rastreador do librosa")
            y, sr = librosa.load(str(origem), sr=22050, mono=True)
            _, batidas = librosa.beat.beat_track(y=y, sr=sr, units="time")
            fortes = []
            fonte += " / librosa"
    finally:
        if tmp.exists():
            tmp.unlink()
    return np.asarray(batidas, float), np.asarray(fortes, float), fonte


def transcrever_stem(arquivo, papel, nome_exib, grade, por_compasso, destino,
                     base, titulo_musica, dispositivo):
    import librosa
    aj = AJUSTES[papel]
    y, sr = librosa.load(str(arquivo), sr=16000, mono=True)
    midi, voz, desvio = f0_quadros(y, sr, aj, dispositivo)
    fracao = float(voz.mean())
    if fracao < 0.02:
        return {"aviso": "sem linha melodica detectada", "notas": 0}
    notas = corrigir_oitavas(segmentar(midi, voz, aj))
    if not notas:
        return {"aviso": "nenhuma nota", "notas": 0}
    q = quantizar(notas, grade)
    partitura, k = montar_partitura(q, nome_exib, papel, grade, por_compasso,
                                    f"{titulo_musica} - {nome_exib}")
    xml = destino / f"{base}.musicxml"
    partitura.write("musicxml", fp=str(xml))
    gerar_midi(notas, papel, destino / f"{base}.mid")
    arquivos = {"musicxml": xml.name, "mid": f"{base}.mid"}
    avisos, marca = [], ""
    try:
        marca = gerar_svg_timemap(xml, destino / f"{base}.svg",
                                  destino / f"{base}.timemap.json")
        arquivos["svg"] = f"{base}.svg"
        arquivos["timemap"] = f"{base}.timemap.json"
    except Exception as e:
        avisos.append(f"partitura na tela nao gerada: {e}")
    try:
        gerar_pdf(xml, destino / f"{base}.pdf")
        arquivos["pdf"] = f"{base}.pdf"
    except Exception as e:
        avisos.append(f"PDF nao gerado: {e}")
    conf = confianca_compassos(y, sr, q, grade, por_compasso)
    validas = [c for c in conf if c is not None]
    return {
        "nome": nome_exib, "papel": papel, "arquivos": arquivos,
        "tonalidade": f"{k.tonic.name} {k.mode}",
        "notas": len(q), "fracao_com_som": round(fracao, 3),
        "confianca": round(float(np.mean(validas)), 3) if validas else 0.0,
        "confianca_compassos": [None if c is None else round(c, 3) for c in conf],
        "modelo": "torchcrepe-full",
        "afinacao_cents": int(round(desvio * 100)),
        "hash": marca,
        "aviso": "; ".join(avisos),
    }


def transcrever_musica(pasta, limpar_nome, dispositivo, refazer):
    destino = pasta / "partituras"
    destino.mkdir(exist_ok=True)
    jpath = destino / "partitura.json"
    if jpath.exists() and not refazer:
        print("   ja tem partitura, pulando")
        return "pulada"

    escolhidos, vistos = {}, set()
    for f in sorted(pasta.iterdir()):
        if not f.is_file() or f.suffix.lower() not in EXT:
            continue
        chave = f.stem.lower()
        if chave in PAPEIS:
            escolhidos[chave] = f
    # voz principal ganha da voz inteira
    if "vocal_principal" in escolhidos:
        escolhidos.pop("vocal", None)
    if not escolhidos:
        print("   sem faixa de voz nem de baixo, pulando")
        return "sem_faixas"

    t0 = time.time()
    batidas, fortes, fonte = grade_da_musica(pasta, limpar_nome, dispositivo)
    import librosa
    ref = next(iter(escolhidos.values()))
    duracao = librosa.get_duration(path=str(ref))
    grade, por_compasso = montar_grade(batidas, fortes, duracao)
    bpm = 60.0 / float(np.median(np.diff(batidas)))
    print(f"   batidas por {fonte}: {len(batidas)} | ~{bpm:.0f} BPM | "
          f"{por_compasso}/4 | {time.time()-t0:.0f} s")

    stems = {}
    for chave, arq in escolhidos.items():
        papel, nome_exib = PAPEIS[chave]
        print(f"   {chave} ({papel})...", end=" ", flush=True)
        t1 = time.time()
        try:
            r = transcrever_stem(arq, papel, nome_exib, grade, por_compasso,
                                 destino, chave, pasta.name, dispositivo)
        except Exception as e:
            import traceback; traceback.print_exc()
            r = {"aviso": f"falhou: {e}", "notas": 0}
        stems[chave] = r
        if r.get("notas"):
            print(f"{r['notas']} notas, confianca {r['confianca']:.2f}, "
                  f"{r['tonalidade']}, {time.time()-t1:.0f} s"
                  + (f"  [{r['aviso']}]" if r.get("aviso") else ""))
        else:
            print(r.get("aviso", "nada"))

    info = {
        "versao": VERSAO_FORMATO,
        "gerado": datetime.now().isoformat(timespec="seconds"),
        "musica": pasta.name,
        "bpm": round(bpm, 1),
        "compasso": [por_compasso, 4],
        "fonte_batidas": fonte,
        "batidas": [round(float(t), 4) for t in grade],
        "stems": {k: v for k, v in stems.items() if v.get("notas")},
        "ignorados": {k: v.get("aviso", "") for k, v in stems.items()
                      if not v.get("notas")},
    }
    jpath.write_text(json.dumps(info, ensure_ascii=False), encoding="utf-8")
    return "feita" if info["stems"] else "vazia"


def principal():
    modo = globals().get("PARTITURA", "voz_e_baixo")
    refazer = bool(globals().get("REFAZER", False))
    if modo == "nenhuma":
        print("PARTITURA = nenhuma na celula anterior; nada a fazer aqui.")
        return

    etapa("Instalando")
    instalar()
    import torch
    dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"   dispositivo: {dispositivo}")

    limpar = globals().get("limpar_nome") or (lambda t: t)

    etapa("Transcrevendo")
    pastas = [p for p in sorted(SAIDA.iterdir()) if p.is_dir()] \
        if SAIDA.exists() else []
    if not pastas:
        raise SystemExit("nenhuma musica em /content/saida; rode a celula anterior")
    res = {"feita": 0, "pulada": 0, "falhou": []}
    for i, p in enumerate(pastas, 1):
        print(f"\n[{i}/{len(pastas)}] {p.name}")
        try:
            r = transcrever_musica(p, limpar, dispositivo, refazer)
            res[r] = res.get(r, 0) + 1
        except Exception as e:
            import traceback; traceback.print_exc()
            res["falhou"].append(p.name)
    print(f"\n   {res.get('feita', 0)} com partitura, {res['pulada']} puladas"
          + (f", {len(res['falhou'])} com erro: {', '.join(res['falhou'])}"
             if res["falhou"] else ""))

    etapa("Baixando")
    total = sum(f.stat().st_size for f in SAIDA.rglob("*") if f.is_file()) / 1e6
    if total > 400:
        print("   AVISO: acima de 400 MB o download costuma falhar.")
        print("   Se cair, rode de novo com FORMATO = MP3 na celula anterior.")
    shutil.make_archive("/content/stems", "zip", str(SAIDA))
    tam = Path("/content/stems.zip").stat().st_size / 1e6
    print(f"   stems.zip ({tam:.0f} MB) — o download comeca sozinho.")
    if "google.colab" in sys.modules:
        from google.colab import files as _f
        _f.download("/content/stems.zip")
    print(f"\n{'='*62}")
    print("  Pronto. Descompacte o stems.zip dentro da pasta 'saida'")
    print("  do KaraokeSep e abra o festa.bat.")
    print(f"{'='*62}")


if __name__ == "__main__":
    principal()

---

## Para separar mais músicas

Apague a pasta das músicas antigas e rode a célula de novo:

```python
import shutil
shutil.rmtree("/content/entrada", ignore_errors=True)
```

Ou marque `REFAZER` se quiser reprocessar as mesmas em outro modo.

O que já foi separado é pulado automaticamente, então rodar de novo depois de
uma queda continua de onde parou.

## Sobre a separação de vozes

Só faz sentido em dueto com timbres diferentes e trechos alternados. Aí o
modelo quase só precisa decidir quem está cantando em cada momento.

Onde não funciona: harmonia em terças, refrão com os dois no uníssono, duas
vozes do mesmo registro. O SDR de 7,29 já avisa — o modelo de baixo, para
comparação, chega a 14,6.

## Se der problema

**Parou dizendo que não há placa** — ligue a T4 em `Ambiente de execução` →
`Alterar o tipo de ambiente de execução`.

**O Colab desconectou** — a sessão grátis cai por inatividade e tem teto de
horas por dia. Rode a célula de novo; a instalação e as músicas enviadas se
perdem, mas o fluxo é o mesmo.

**Upload muito lento** — é a sua internet subindo, uns 30 MB por música em
FLAC. Se estiver inviável, mande MP3 320, aceitando a perda na fonte.

**Download falhou** — rode com `FORMATO = MP3`, ou separe menos músicas por vez.

**Linhas vermelhas sobre `libcublas` ou `libcudart`** — pode ignorar. É o
`onnxruntime` reclamando de bibliotecas que os modelos usados aqui não precisam.

**Faltou alguma faixa no modo festa** — rode o `diagnostico.bat` no seu PC.